# Spark SQL Sandbox (spylon-kernel)

Use this notebook with the **spylon-kernel** for pure Spark SQL via `%%sql` magic.

In [ ]:
%%init_spark
launcher.master = "local[*]"
launcher.conf.set("spark.app.name", "SparkSQL-Sandbox")
launcher.conf.set("spark.sql.warehouse.dir", "/workspace/data/lake")
launcher.conf.set("spark.jars", "/opt/spark/jars/mssql-jdbc-12.4.2.jre11.jar")

In [ ]:
%%sql
CREATE OR REPLACE TEMPORARY VIEW bronze_customers
USING parquet OPTIONS (path '/workspace/data/lake/bronze/customers')

In [ ]:
%%sql
CREATE OR REPLACE TEMPORARY VIEW bronze_orders
USING parquet OPTIONS (path '/workspace/data/lake/bronze/orders')

In [ ]:
%%sql
SELECT c.name, c.region,
       COUNT(o.order_id) AS num_orders,
       SUM(o.quantity * o.unit_price) AS total_spend
FROM bronze_customers c
JOIN bronze_orders o ON c.customer_id = o.customer_id
GROUP BY c.name, c.region
ORDER BY total_spend DESC

In [ ]:
%%sql
-- Direct JDBC query against MSSQL via Spark SQL
CREATE OR REPLACE TEMPORARY VIEW mssql_orders
USING org.apache.spark.sql.jdbc
OPTIONS (
    url 'jdbc:sqlserver://mssql:1433;databaseName=fabric_demo;encrypt=false;trustServerCertificate=true',
    dbtable 'dbo.orders',
    user 'sa',
    password 'FabricLocal#2024',
    driver 'com.microsoft.sqlserver.jdbc.SQLServerDriver'
);

SELECT product, SUM(quantity) as total_qty
FROM mssql_orders GROUP BY product ORDER BY total_qty DESC